# Tugas Besar 1 — Klasifikasi Biner dengan Perceptron dan ADALINE
**Mata kuliah:** Jaringan Syaraf Tiruan (IT6053) — FTI UKDW

| | |
|---|---|
| **Kelompok** | `XX` ⟨isi⟩ |
| **Anggota (Nama / NIM)** | ⟨isi nama 1 / NIM⟩ · ⟨isi nama 2 / NIM⟩ · ⟨isi nama 3 / NIM (jika ada)⟩ |
| **Dataset** | KFC Sales — `dataset.csv` (turunan dari `kfc_sales_data.xlsx`) |
| **Kelas positif** | **High Revenue = +1**, Low Revenue = −1 |
| **Seed** | 42 |

> **Cara menjalankan:** letakkan `dataset.csv` di folder yang sama dengan notebook ini lalu pilih *Run All*.
> Jika hanya ada `kfc_sales_data.xlsx`, sel pemuatan data akan mengonversinya menjadi `dataset.csv` secara otomatis.

### Tujuan
Melanjutkan kasus Tugas 1 menjadi eksperimen klasifikasi dua kelas dengan **Perceptron** dan **ADALINE (Delta Rule)** yang diimplementasikan sendiri dengan NumPy, lalu menjelaskan hubungan antara masalah, fitur, label, model neuron, proses pembelajaran, dan hasil evaluasi.

### Sumber data, izin, dan keterkaitan dengan Tugas 1 ⟨isi⟩
- **Sumber dataset & izin penggunaan:** ⟨tulis sumber/URL, lisensi, dan pernyataan bahwa data boleh dipublikasikan⟩
- **Keterkaitan dengan Tugas 1:** ⟨jelaskan karakteristik data yang diidentifikasi pada Tugas 1 dan mengapa cocok untuk klasifikasi dua kelas⟩

### Bantuan AI dan verifikasi ⟨isi⟩
⟨Sebutkan bagian mana yang dibantu AI (mis. kerangka kode, penjelasan) dan bagaimana kelompok memverifikasinya (mis. menjalankan ulang, menghitung manual, membandingkan dengan sumber).⟩

## 1. Rumusan Masalah (Supervised Learning)

Memprediksi apakah sebuah transaksi KFC termasuk **High Revenue** atau **Low Revenue** dari karakteristik transaksinya.

- **Fitur (x):** `hour`, `items_count` (numerik) dan `day_of_week`, `country`, `city`, `channel`, `payment_method` (kategorikal → one-hot).
- **Target (y):** `+1` jika `revenue_usd` ≥ median, `−1` jika di bawah median.
- **Alasan dua kelas dari median:** median memberi dua kelas yang hampir seimbang sehingga akurasi mudah ditafsirkan dan tidak didominasi satu kelas.
- **Pencegahan data leakage:** `subtotal_usd`, `discount_usd`, `delivery_fee_usd`, dan `revenue_usd` **tidak dipakai sebagai fitur** karena target dibentuk langsung dari revenue. `transaction_id`, `date`, `store_id`, dan `menu_item` juga tidak dipakai (identifier / tidak dipilih).

In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED   = 42
EPOCHS = 50
ETAS   = [0.001, 0.01, 0.1]
np.random.seed(SEED)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 110
print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)

In [ ]:
DATA_PATH = "dataset.csv"

# Konversi sekali jalan: xlsx -> csv (hanya jika dataset.csv belum ada)
if not os.path.exists(DATA_PATH):
    for cand in ["kfc_sales_data.xlsx", "/content/kfc_sales_data.xlsx"]:
        if os.path.exists(cand):
            pd.read_excel(cand).to_csv(DATA_PATH, index=False)
            print("dataset.csv dibuat dari", cand)
            break

df = pd.read_csv(DATA_PATH)
print("Jumlah observasi:", df.shape[0], "| jumlah kolom:", df.shape[1])
print("Kolom:", df.columns.tolist())
df.head()

In [ ]:
print("Missing value per kolom:")
print(df.isna().sum()[df.isna().sum() > 0] if df.isna().any().any() else "Tidak ada missing value")
print("\nJumlah baris duplikat:", df.duplicated().sum())
print("\nTipe data:"); print(df.dtypes)
print("\nRingkasan numerik:"); display(df[["hour","items_count","subtotal_usd","discount_usd","delivery_fee_usd","revenue_usd"]].describe().round(2))
print("\nJumlah kategori unik:")
print(df[["day_of_week","country","city","channel","payment_method"]].nunique())

## 2. Pembentukan Label (−1 / +1)

Batas kelas adalah median `revenue_usd`. Kelas positif = **High Revenue (+1)**.

> Catatan desain: median dihitung dari seluruh data karena ia mendefinisikan *masalahnya* (apa arti "high"), bukan parameter model. Seluruh parameter **pra-pemrosesan** (standardisasi) tetap dipelajari hanya dari data latih.

In [ ]:
median_revenue = df["revenue_usd"].median()
df["label"] = np.where(df["revenue_usd"] >= median_revenue, 1, -1)   # +1 = High, -1 = Low

print(f"Median revenue_usd: {median_revenue:.2f}")
dist = df["label"].value_counts().rename({1: "High Revenue (+1)", -1: "Low Revenue (-1)"})
print(pd.DataFrame({"jumlah": dist, "proporsi": (dist / len(df)).round(4)}))

## 3. Fitur, Pembagian Data, dan Standardisasi

1. Fitur kategorikal → one-hot (kolom 0/1). Dua fitur numerik (`hour`, `items_count`) ada di kolom 0 dan 1.
2. Split **80:20**, **stratifikasi** kelas, `random_state=42`. **Pembagian yang sama dipakai oleh semua model dan semua konfigurasi.**
3. Standardisasi (z-score) **hanya pada dua fitur numerik**, `mean` dan `std` dipelajari dari **data latih saja**, lalu diterapkan ke data uji. Kolom one-hot dibiarkan 0/1.
4. Dua versi data disimpan: `*_std` (terstandardisasi) dan `*_raw` (tanpa standardisasi) — keduanya dengan baris dan urutan sampel yang identik.

In [ ]:
numeric_features     = ["hour", "items_count"]
categorical_features = ["day_of_week", "country", "city", "channel", "payment_method"]

X_df = pd.get_dummies(df[numeric_features + categorical_features],
                      columns=categorical_features, dtype=float)
feature_names = X_df.columns.tolist()
X = X_df.values
y = df["label"].values
print("Jumlah fitur setelah one-hot:", X.shape[1])

idx_all = np.arange(len(df))
idx_train, idx_test = train_test_split(idx_all, test_size=0.20, stratify=y, random_state=SEED)

Xtr_raw, Xte_raw = X[idx_train].copy(), X[idx_test].copy()
y_train, y_test  = y[idx_train], y[idx_test]

scaler = StandardScaler().fit(Xtr_raw[:, :2])          # parameter HANYA dari data latih
Xtr_std, Xte_std = Xtr_raw.copy(), Xte_raw.copy()
Xtr_std[:, :2] = scaler.transform(Xtr_raw[:, :2])
Xte_std[:, :2] = scaler.transform(Xte_raw[:, :2])

print("Train:", Xtr_std.shape, "| Test:", Xte_std.shape)
print("Distribusi train (-1, +1):", [int((y_train == c).sum()) for c in (-1, 1)])
print("Distribusi test  (-1, +1):", [int((y_test  == c).sum()) for c in (-1, 1)])
print("Mean/Std data latih (hour, items_count):", scaler.mean_.round(3), scaler.scale_.round(3))

## 4. Model Neuron dan Aturan Pembelajaran

**Neuron & keputusan (kedua model):**
$$z = \mathbf{w}^\top \mathbf{x} + b,\qquad \hat y = \begin{cases}+1 & z \ge 0\\ -1 & z < 0\end{cases}$$

**Perceptron** (per sampel, urutan data tetap, inisialisasi $\mathbf{w}=\mathbf{0}$, $b=0$):
$$\Delta = \eta\,(y-\hat y),\qquad \mathbf{w}\leftarrow \mathbf{w}+\Delta\,\mathbf{x},\qquad b\leftarrow b+\Delta$$
Error yang dipakai berasal dari **keluaran ambang** $\hat y\in\{-1,+1\}$, sehingga bobot hanya berubah saat salah klasifikasi (saat salah, $\Delta=\pm 2\eta$). Dicatat: jumlah pembaruan per epoch.

**ADALINE / Delta Rule** (batch gradient descent, inisialisasi nol):
$$e = y - z,\qquad \mathbf{w}\leftarrow \mathbf{w}+\eta\,\frac{X^\top e}{n},\qquad b\leftarrow b+\eta\,\overline{e}$$
Error memakai **keluaran linear** $z$ (bukan keluaran ambang); ambang hanya dipakai saat prediksi.

**Fungsi objektif:** $J=\dfrac{1}{2n}\sum_i (y_i-z_i)^2$ (dan $\text{MSE}=2J$). **$J$ dicatat sebelum pembaruan pada epoch tersebut**; kolom "J akhir" pada tabel hasil dihitung ulang dengan bobot setelah pembaruan terakhir.

Jika $J$ menjadi `inf`/`NaN`, pelatihan dihentikan dan epoch kejadian dicatat.

In [ ]:
class Perceptron:
    # Perceptron: update per sampel, Delta = eta * (y - y_hat). Label dalam {-1, +1}.
    def __init__(self, eta=0.01, epochs=50):
        self.eta, self.epochs = eta, epochs

    def net_input(self, X):
        return X @ self.w + self.b

    def predict(self, X):
        return np.where(self.net_input(X) >= 0, 1, -1)

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])          # inisialisasi nol
        self.b = 0.0
        self.updates_, self.train_acc_ = [], []
        self.w_hist_, self.b_hist_ = [], []    # snapshot per epoch
        for _ in range(self.epochs):
            n_updates = 0
            for xi, yi in zip(X, y):           # urutan data TETAP (tanpa shuffle)
                z = xi @ self.w + self.b
                y_hat = 1 if z >= 0 else -1
                delta = self.eta * (yi - y_hat)
                if delta != 0.0:
                    self.w += delta * xi
                    self.b += delta
                    n_updates += 1
            self.updates_.append(n_updates)
            self.train_acc_.append(float(np.mean(self.predict(X) == y)))
            self.w_hist_.append(self.w.copy()); self.b_hist_.append(self.b)
        return self


class Adaline:
    # ADALINE: batch gradient descent pada output linear. J = (1/2n) sum (y - z)^2.
    def __init__(self, eta=0.01, epochs=50):
        self.eta, self.epochs = eta, epochs

    def net_input(self, X):
        return X @ self.w + self.b

    def predict(self, X):
        return np.where(self.net_input(X) >= 0, 1, -1)

    def cost(self, X, y):
        e = y - self.net_input(X)
        return float(np.sum(e ** 2) / (2 * len(y)))

    def fit(self, X, y):
        n = len(y)
        self.w = np.zeros(X.shape[1])
        self.b = 0.0
        self.loss_, self.train_acc_ = [], []   # loss_ = J SEBELUM pembaruan epoch tsb
        self.stopped_epoch_ = None
        with np.errstate(all="ignore"):
            for epoch in range(1, self.epochs + 1):
                z = X @ self.w + self.b
                e = y - z                                   # error dari output LINEAR
                J = float(np.sum(e ** 2) / (2 * n))
                if not np.isfinite(J):                      # inf / NaN -> hentikan
                    self.stopped_epoch_ = epoch
                    break
                self.loss_.append(J)
                self.w = self.w + self.eta * (X.T @ e) / n
                self.b = self.b + self.eta * e.mean()
                self.train_acc_.append(float(np.mean(self.predict(X) == y)))
            self.final_loss_ = self.cost(X, y)              # J dengan bobot akhir
        return self

## 5. Perhitungan Manual Dua Langkah (untuk Lampiran Laporan)

Agar angka mudah diperiksa dengan kalkulator, perhitungan manual memakai **2 fitur** (`hour` dan `items_count` terstandardisasi), $\eta=0.1$, dan 4 sampel latih pertama (2 Low + 2 High, urutan Low, High, Low, High).
Sel di bawah mencetak tiap langkah, lalu **memverifikasi bahwa hasilnya sama dengan kelas `Perceptron` dan `Adaline`** (kode ⇔ hitungan manual konsisten).

- **Perceptron:** dua sampel berurutan (sampel 1 dan 2).
- **ADALINE:** dua iterasi batch pada 4 sampel.

In [ ]:
lo = np.where(y_train == -1)[0][:2]
hi = np.where(y_train ==  1)[0][:2]
idx_m = np.array([lo[0], hi[0], lo[1], hi[1]])
Xm, ym = Xtr_std[idx_m][:, :2], y_train[idx_m]
eta_m = 0.1
np.set_printoptions(precision=4, suppress=True)

print("Sampel manual (hour_std, items_std) dan label:")
for i, (xi, yi) in enumerate(zip(Xm, ym), 1):
    print(f"  x{i} = {xi.round(4)}  y = {yi:+d}")

# ---------------- Perceptron: 2 sampel ----------------
print("\n=== PERCEPTRON (eta = 0.1), w0 = [0, 0], b0 = 0 ===")
w, b = np.zeros(2), 0.0
for i in range(2):
    z = Xm[i] @ w + b
    y_hat = 1 if z >= 0 else -1
    delta = eta_m * (ym[i] - y_hat)
    print(f"Sampel {i+1}: z = {z:+.4f} -> y_hat = {y_hat:+d}, y = {ym[i]:+d}, Delta = {eta_m}*({ym[i]}-({y_hat})) = {delta:+.4f}")
    w = w + delta * Xm[i]; b = b + delta
    print(f"          w = {w.round(4)}, b = {b:+.4f}")
w_p_manual, b_p_manual = w.copy(), b

# ---------------- ADALINE: 2 iterasi batch ----------------
print("\n=== ADALINE (eta = 0.1, n = 4), w0 = [0, 0], b0 = 0 ===")
w, b, n = np.zeros(2), 0.0, len(ym)
for it in (1, 2):
    z = Xm @ w + b
    e = ym - z
    J = np.sum(e ** 2) / (2 * n)
    dw = eta_m * (Xm.T @ e) / n
    db = eta_m * e.mean()
    print(f"Iterasi {it}: z = {z.round(4)}")
    print(f"           e = y - z = {e.round(4)}, J(sebelum update) = {J:.4f}")
    print(f"           dw = eta*X^T e/n = {dw.round(4)}, db = eta*mean(e) = {db:+.4f}")
    w, b = w + dw, b + db
    print(f"           w = {w.round(4)}, b = {b:+.4f}")
w_a_manual, b_a_manual = w.copy(), b

# ---------------- verifikasi dengan kelas ----------------
p_chk = Perceptron(eta=eta_m, epochs=1).fit(Xm[:2], ym[:2])
a_chk = Adaline(eta=eta_m, epochs=2).fit(Xm, ym)
assert np.allclose(p_chk.w, w_p_manual) and np.isclose(p_chk.b, b_p_manual), "Perceptron: kode != manual"
assert np.allclose(a_chk.w, w_a_manual) and np.isclose(a_chk.b, b_a_manual), "ADALINE: kode != manual"
print("\nVerifikasi: bobot/bias akhir hitungan manual == hasil kelas Perceptron & Adaline  OK")
print("Perceptron akhir: w =", w_p_manual.round(4), " b =", round(b_p_manual, 4))
print("ADALINE    akhir: w =", w_a_manual.round(4), " b =", round(b_a_manual, 4))

## 6. Rancangan Eksperimen (ditetapkan sebelum melihat data uji)

| # | Model | η | Epoch | Standardisasi |
|---|---|---|---|---|
| 1–3 | Perceptron | 0,001 · 0,01 · 0,1 | 50 | Ya |
| 4–6 | ADALINE | 0,001 · 0,01 · 0,1 | 50 | Ya |
| 7 | Perceptron | 0,01 | 50 | **Tidak** |
| 8 | ADALINE | 0,01 | 50 | **Tidak** |

- Split, inisialisasi (nol), dan urutan sampel **sama** di semua pelatihan.
- **Tidak ada seleksi parameter dengan data uji.** Seluruh 8 konfigurasi dilaporkan apa adanya; data uji dipakai hanya untuk pelaporan akhir.
- **Konfigurasi yang dibahas secara rinci** (bobot, decision boundary): **η = 0,01 dengan standardisasi** — ditetapkan sebelum evaluasi, bukan karena hasil uji terbaik.
- Jika ADALINE menghasilkan `inf`/`NaN`, pelatihan dihentikan dan epoch kejadian dicatat.

In [ ]:
CONFIGS = ([("Perceptron", eta, True) for eta in ETAS] +
           [("ADALINE",    eta, True) for eta in ETAS] +
           [("Perceptron", 0.01, False), ("ADALINE", 0.01, False)])
DISCUSSED = 0.01   # konfigurasi yang dibahas: eta = 0.01, standardisasi = Ya

models = {}
for name, eta, std in CONFIGS:
    Xtr_ = Xtr_std if std else Xtr_raw
    m = Perceptron(eta, EPOCHS) if name == "Perceptron" else Adaline(eta, EPOCHS)
    t0 = time.time()
    m.fit(Xtr_, y_train)
    models[(name, eta, std)] = m
    print(f"{name:10s} eta={eta:<6} std={'Ya ' if std else 'Tdk'}  selesai dalam {time.time()-t0:5.1f} dtk")

## 7. Hasil

### 7.1 Tabel seluruh konfigurasi
- **Update/error epoch terakhir** (Perceptron) = jumlah pembaruan akibat salah klasifikasi pada epoch ke-50; **J akhir** (ADALINE) = $J$ dengan bobot akhir.
- Akurasi latih/uji dihitung dengan bobot akhir (epoch ke-50 atau epoch terakhir sebelum berhenti).

In [ ]:
def get_data(std):
    return (Xtr_std, Xte_std) if std else (Xtr_raw, Xte_raw)

def evaluate(m, std):
    Xtr_, Xte_ = get_data(std)
    if not np.all(np.isfinite(m.w)) or not np.isfinite(m.b):
        return dict(acc_tr=np.nan, acc_te=np.nan, pred=None)
    ptr, pte = m.predict(Xtr_), m.predict(Xte_)
    return dict(acc_tr=accuracy_score(y_train, ptr), acc_te=accuracy_score(y_test, pte), pred=pte)

rows, evals = [], {}
for (name, eta, std), m in models.items():
    ev = evaluate(m, std); evals[(name, eta, std)] = ev
    if name == "Perceptron":
        last_err = m.updates_[-1]
        status = "konvergen (0 update)" if last_err == 0 else "tidak konvergen (masih ada salah klasifikasi)"
        j_final = np.nan
    else:
        last_err = np.nan
        j_final = m.final_loss_
        if m.stopped_epoch_ is not None:
            status = f"loss inf/NaN pada epoch {m.stopped_epoch_} (dihentikan)"
        elif m.loss_[-1] > m.loss_[0]:
            status = "DIVERGEN (J naik)"
        else:
            rel = abs(m.loss_[-1] - m.loss_[-11]) / max(m.loss_[-11], 1e-12) if len(m.loss_) > 10 else np.nan
            status = "J turun, hampir konstan" if rel < 1e-3 else "J turun, masih menurun"
    rows.append({"Model": name, "η": eta, "Epoch": len(m.updates_) if name=="Perceptron" else len(m.loss_),
                 "Standardisasi": "Ya" if std else "Tidak",
                 "Akurasi latih": ev["acc_tr"], "Akurasi uji": ev["acc_te"],
                 "Update epoch terakhir (Perc.)": last_err, "J akhir (ADALINE)": j_final, "Status": status})

results = pd.DataFrame(rows)
display(results.style.format({"Akurasi latih": "{:.4f}", "Akurasi uji": "{:.4f}",
                              "Update epoch terakhir (Perc.)": "{:.0f}", "J akhir (ADALINE)": "{:.4f}", "η": "{:g}"},
                             na_rep="-").hide(axis="index"))

### 7.2 Kondisi pada epoch ke-10, ke-30, dan ke-50 (dari riwayat **pelatihan**)
Hanya memakai riwayat data latih (bukan data uji), sehingga tidak ada kebocoran evaluasi.

In [ ]:
snap = []
for (name, eta, std), m in models.items():
    for ep in (10, 30, 50):
        if name == "Perceptron":
            val = m.updates_[ep-1] if ep <= len(m.updates_) else np.nan
            acc = m.train_acc_[ep-1] if ep <= len(m.train_acc_) else np.nan
            snap.append({"Model": name, "η": eta, "Std": "Ya" if std else "Tidak", "Epoch": ep,
                         "Update salah klasifikasi": val, "J (sebelum update)": np.nan, "Akurasi latih": acc})
        else:
            val = m.loss_[ep-1] if ep <= len(m.loss_) else np.nan
            acc = m.train_acc_[ep-1] if ep <= len(m.train_acc_) else np.nan
            snap.append({"Model": name, "η": eta, "Std": "Ya" if std else "Tidak", "Epoch": ep,
                         "Update salah klasifikasi": np.nan, "J (sebelum update)": val, "Akurasi latih": acc})
snap = pd.DataFrame(snap)
display(snap.style.format({"Update salah klasifikasi": "{:.0f}", "J (sebelum update)": "{:.4f}",
                           "Akurasi latih": "{:.4f}", "η": "{:g}"}, na_rep="-").hide(axis="index"))

### 7.3 Kurva pembelajaran (hingga epoch ke-50)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))

for eta in ETAS:
    m = models[("Perceptron", eta, True)]
    ax[0].plot(range(1, len(m.updates_) + 1), m.updates_, label=f"η={eta:g} (std)", alpha=.85)
m = models[("Perceptron", 0.01, False)]
ax[0].plot(range(1, len(m.updates_) + 1), m.updates_, "k--", label="η=0.01 (tanpa std)")
ax[0].set_title("Perceptron: jumlah pembaruan salah klasifikasi / epoch")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Jumlah pembaruan"); ax[0].legend()

for eta in ETAS:
    m = models[("ADALINE", eta, True)]
    ax[1].plot(range(1, len(m.loss_) + 1), m.loss_, label=f"η={eta:g}")
ax[1].set_title("ADALINE (standardisasi): J per epoch")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("J (sebelum update)"); ax[1].legend()

m = models[("ADALINE", 0.01, False)]
ax[2].plot(range(1, len(m.loss_) + 1), m.loss_, "k--")
ax[2].set_yscale("log")
ax[2].set_title("ADALINE tanpa standardisasi (η=0.01): J per epoch (skala log)")
ax[2].set_xlabel("Epoch"); ax[2].set_ylabel("J (log)")
plt.tight_layout(); plt.show()

print("Kesalahan latih setelah epoch terakhir:")
for (name, eta, std), m in models.items():
    ev = evals[(name, eta, std)]
    print(f"  {name:10s} η={eta:<6g} std={'Ya ' if std else 'Tdk'} -> error latih = {1-ev['acc_tr']:.4f}")

### 7.4 Confusion matrix data uji (urutan label: **−1 = Low**, **+1 = High**; baris = aktual, kolom = prediksi)
Precision, recall, dan F1 dihitung untuk **kelas positif (+1, High Revenue)**.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 8))
prf_rows = []
for ax, ((name, eta, std), m) in zip(axes.ravel(), models.items()):
    pred = evals[(name, eta, std)]["pred"]
    title = f"{name} η={eta:g} {'std' if std else 'tanpa std'}"
    if pred is None:
        ax.axis("off"); ax.set_title(title + "\n(bobot inf/NaN)"); continue
    cm = confusion_matrix(y_test, pred, labels=[-1, 1])
    ConfusionMatrixDisplay(cm, display_labels=["Low (-1)", "High (+1)"]).plot(ax=ax, colorbar=False, values_format="d")
    ax.set_title(title, fontsize=10)
    p, r, f, _ = precision_recall_fscore_support(y_test, pred, labels=[1], average=None, zero_division=0)
    prf_rows.append({"Model": name, "η": eta, "Std": "Ya" if std else "Tidak",
                     "TN": cm[0,0], "FP": cm[0,1], "FN": cm[1,0], "TP": cm[1,1],
                     "Precision (+1)": p[0], "Recall (+1)": r[0], "F1 (+1)": f[0]})
plt.tight_layout(); plt.show()

prf = pd.DataFrame(prf_rows)
display(prf.style.format({"Precision (+1)": "{:.4f}", "Recall (+1)": "{:.4f}", "F1 (+1)": "{:.4f}", "η": "{:g}"}).hide(axis="index"))

### 7.5 Bobot dan bias akhir (konfigurasi yang dibahas: η = 0,01, standardisasi)

In [ ]:
mp = models[("Perceptron", DISCUSSED, True)]
ma = models[("ADALINE",    DISCUSSED, True)]
w_tbl = pd.DataFrame({"fitur": feature_names, "w Perceptron": mp.w, "w ADALINE": ma.w}).set_index("fitur")
display(w_tbl.round(4))
print(f"Bias akhir  Perceptron: b = {mp.b:+.4f}")
print(f"Bias akhir  ADALINE   : b = {ma.b:+.4f}")
print("\n5 fitur dengan |bobot| terbesar:")
print("  Perceptron:", w_tbl["w Perceptron"].abs().nlargest(5).index.tolist())
print("  ADALINE   :", w_tbl["w ADALINE"].abs().nlargest(5).index.tolist())

### 7.6 Decision boundary (proyeksi dua fitur)
Model memiliki **30 fitur**, sehingga batas keputusan sebenarnya berupa hyperplane 30 dimensi. Gambar di bawah adalah **proyeksi pada bidang `hour` × `items_count` (terstandardisasi)** dengan **28 fitur lain ditetapkan pada rata-rata data latih**. Garis menunjukkan $z=0$. Ini hanya ilustrasi; titik-titik sebenarnya berbeda pada fitur lain sehingga banyak titik berada di sisi "salah" dari garis proyeksi.

In [ ]:
rng = np.random.default_rng(SEED)
sel = rng.choice(len(Xte_std), size=2000, replace=False)
means_rest = Xtr_std[:, 2:].mean(axis=0)

g1, g2 = np.meshgrid(np.linspace(-2.2, 2.2, 300), np.linspace(-2.2, 2.2, 300))
def z_plane(m):
    return m.w[0] * g1 + m.w[1] * g2 + (m.b + means_rest @ m.w[2:])

fig, ax = plt.subplots(figsize=(7, 5.5))
jit = rng.normal(0, 0.03, size=(len(sel), 2))
for cls, col, lab in [(-1, "tab:blue", "Low (-1)"), (1, "tab:red", "High (+1)")]:
    mk = y_test[sel] == cls
    ax.scatter(Xte_std[sel][mk, 0] + jit[mk, 0], Xte_std[sel][mk, 1] + jit[mk, 1], s=8, alpha=.35, c=col, label=lab)
c1 = ax.contour(g1, g2, z_plane(mp), levels=[0], colors="k", linewidths=2)
c2 = ax.contour(g1, g2, z_plane(ma), levels=[0], colors="g", linewidths=2, linestyles="--")
ax.plot([], [], "k-", label="Perceptron (z=0)"); ax.plot([], [], "g--", label="ADALINE (z=0)")
ax.set_xlabel("hour (standardisasi)"); ax.set_ylabel("items_count (standardisasi)")
ax.set_title("Proyeksi decision boundary (fitur lain = rata-rata latih)")
ax.legend(loc="best", fontsize=8); plt.show()

## 8. Analisis Pendukung Pembahasan
Sel-sel berikut menghasilkan bukti kuantitatif untuk empat pertanyaan pembahasan di Bagian 9.

In [ ]:
# (a) Indikasi keterpisahan linear: pola data (hanya data LATIH) + perilaku pelatihan
tr = df.iloc[idx_train]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
(tr.groupby("items_count")["label"].apply(lambda s: (s == 1).mean())).plot(kind="bar", ax=ax[0], color="tab:red")
ax[0].set_title("Proporsi High Revenue per items_count (data latih)"); ax[0].set_ylabel("proporsi High"); ax[0].axhline(.5, c="k", ls=":")
(tr.groupby("channel")["label"].apply(lambda s: (s == 1).mean())).plot(kind="bar", ax=ax[1], color="tab:blue")
ax[1].set_title("Proporsi High Revenue per channel (data latih)"); ax[1].axhline(.5, c="k", ls=":")
plt.tight_layout(); plt.show()

n_tr = len(y_train)
print("Perilaku pelatihan Perceptron (standardisasi):")
for eta in ETAS:
    m = models[("Perceptron", eta, True)]
    print(f"  η={eta:<6g} update minimum selama 50 epoch = {min(m.updates_):>6d} "
          f"({min(m.updates_)/n_tr:.2%} dari {n_tr} sampel) | epoch terakhir = {m.updates_[-1]}")
print("-> Jika update minimum jauh di atas 0 dan tidak menurun, data TIDAK dapat dipisahkan secara linear "
      "(Perceptron hanya berhenti memperbarui bila ada hyperplane yang memisahkan sempurna).")

In [ ]:
# (b) Perceptron: apakah mengubah η hanya menskalakan bobot? (init nol, urutan tetap, η konstan > 0)
P = [models[("Perceptron", eta, True)] for eta in ETAS]
ref = P[1]
print("Perbandingan terhadap η = 0.01 (data standardisasi):")
for m in P:
    same_pred = np.array_equal(m.predict(Xte_std), ref.predict(Xte_std))
    diff_w = np.max(np.abs(m.w / m.eta - ref.w / ref.eta))
    same_upd = m.updates_ == ref.updates_
    print(f"  η={m.eta:<6g} prediksi uji identik: {same_pred} | riwayat update identik: {same_upd} | max|w/η - w_ref/η_ref| = {diff_w:.2e}")
print("-> Jika identik: η hanya menskalakan w dan b (urutan prediksi tidak berubah). Hasil yang sama tetap sah.")

In [ ]:
# (c) ADALINE: seberapa dekat gradient descent dengan optimum kuadrat terkecil (pembanding tambahan)
A = np.c_[Xtr_std, np.ones(len(Xtr_std))]
sol, *_ = np.linalg.lstsq(A, y_train, rcond=None)
J_opt = float(np.sum((y_train - A @ sol) ** 2) / (2 * len(y_train)))
pred_ls_te = np.where(np.c_[Xte_std, np.ones(len(Xte_std))] @ sol >= 0, 1, -1)
print(f"J minimum (solusi kuadrat terkecil, data latih standardisasi) = {J_opt:.4f}")
print(f"Akurasi uji solusi kuadrat terkecil (hanya sebagai referensi) = {accuracy_score(y_test, pred_ls_te):.4f}\n")
for eta in ETAS:
    m = models[("ADALINE", eta, True)]
    print(f"  ADALINE η={eta:<6g} J akhir = {m.final_loss_:.4f} | selisih terhadap J minimum = {m.final_loss_ - J_opt:.4f}")
m = models[("ADALINE", 0.01, False)]
print(f"  ADALINE η=0.01 TANPA std: J awal = {m.loss_[0]:.4f}, J akhir = {m.final_loss_:.4g}"
      + (f" (dihentikan pada epoch {m.stopped_epoch_})" if m.stopped_epoch_ else ""))

# batas kestabilan GD: η < 2 / λ_max(XᵀX/n) (dengan kolom bias)
for tag, Xs in [("standardisasi", Xtr_std), ("tanpa standardisasi", Xtr_raw)]:
    Aa = np.c_[Xs, np.ones(len(Xs))]
    lam = np.linalg.eigvalsh(Aa.T @ Aa / len(Aa)).max()
    print(f"  {tag:20s}: λ_max = {lam:9.3f} -> batas stabil η < {2/lam:.4f}")

In [ ]:
# (d) Akurasi sama != proses belajar sama: bandingkan Perceptron vs ADALINE pada konfigurasi yang dibahas
pp = evals[("Perceptron", DISCUSSED, True)]["pred"]; pa = evals[("ADALINE", DISCUSSED, True)]["pred"]
print(f"Akurasi uji Perceptron = {accuracy_score(y_test, pp):.4f} | ADALINE = {accuracy_score(y_test, pa):.4f}")
print(f"Persentase prediksi uji yang berbeda antara kedua model = {np.mean(pp != pa):.2%}")
print(f"Korelasi bobot Perceptron vs ADALINE = {np.corrcoef(mp.w, ma.w)[0,1]:.3f}")
print("Perceptron: bobot terus berubah (osilasi) selama masih ada salah klasifikasi -> bobot akhir bergantung pada sampel terakhir yang salah.")
print("ADALINE   : J menurun mulus menuju minimum karena gradien dari seluruh data (batch).")

## 9. Pembahasan dan Keterbatasan
> **Petunjuk:** bagian ini berupa kerangka. Ganti setiap ⟨…⟩ dengan angka dan kesimpulan **dari output Bagian 7–8 hasil eksekusi Anda sendiri**. Klaim harus didukung bukti (angka/grafik), bukan hanya akurasi uji.

**1. Apakah data menunjukkan indikasi dapat dipisahkan secara linear?**
Pola data (Bagian 8a) menunjukkan ⟨mis. proporsi High naik seiring `items_count` tetapi tiap kategori masih bercampur⟩. Perceptron tetap melakukan ⟨…⟩ pembaruan pada epoch terakhir (minimum ⟨…⟩ per epoch, ⟨…⟩% sampel latih) dan tidak mencapai nol, sehingga ⟨data tidak dapat dipisahkan secara linear / sebaliknya⟩. Karena itu akurasi latih tertinggi yang mungkin bagi model linear dibatasi di sekitar ⟨…⟩.

**2. Bagaimana Perceptron dan ADALINE menggunakan error yang berbeda?**
Perceptron memakai error dari keluaran ambang $(y-\hat y)\in\{0,\pm2\}$ per sampel, sehingga bobot hanya berubah saat salah klasifikasi dan besar koreksinya tidak bergantung pada seberapa jauh $z$ dari batas. ADALINE memakai error dari keluaran linear $e=y-z$ pada seluruh sampel (batch); setiap sampel berkontribusi sebanding dengan jaraknya dari target, sehingga $J$ menurun halus menuju minimum ⟨$J_{min}$ = …⟩. Pada data yang tidak separable, Perceptron berosilasi, sedangkan ADALINE konvergen ke solusi kuadrat terkecil — ini kemungkinan alasan utama perbedaan akurasi uji (Perceptron ⟨…⟩ vs ADALINE ⟨…⟩), bukan sekadar "ADALINE selalu lebih baik".

**3. Pengaruh η, epoch, dan standardisasi**
- **η pada Perceptron:** ⟨hasil Bagian 8b — mis. prediksi identik dan bobot hanya terskala; jelaskan karena inisialisasi nol, urutan tetap, η konstan⟩.
- **η pada ADALINE:** η = 0,001 ⟨konvergen lambat, J masih turun pada epoch 50⟩; η = 0,01 ⟨…⟩; η = 0,1 ⟨konvergen lebih cepat / mendekati J minimum⟩.
- **Epoch:** ⟨bandingkan epoch 10/30/50 dari Tabel 7.2⟩.
- **Standardisasi:** tanpa standardisasi `hour` (0–23) berskala jauh lebih besar dari fitur 0/1 sehingga λ_max besar (Bagian 8c) dan batas stabil η menyempit ⟨→ ADALINE η=0,01 divergen/NaN pada epoch …, Perceptron berubah …⟩.

**4. Apakah akurasi yang sama berarti proses belajar sama?**
⟨Gunakan Bagian 8d: walau akurasi mirip/sama, kurva pembelajaran, bobot akhir, dan sampel yang diprediksi berbeda menunjukkan proses belajar berbeda.⟩ Kesalahan latih nol (jika ada) pun tidak menjamin kinerja sempurna pada data baru.

**Keterbatasan**
- Model linear tidak menangkap interaksi non-linear antar fitur (mis. `items_count` × `channel`).
- Label dibuat dari ambang median sehingga transaksi di dekat median secara intrinsik sulit dipisahkan.
- One-hot `country` dan `city` redundan (kota menentukan negara) sehingga bobot sulit ditafsirkan satu per satu.
- Perceptron tanpa shuffle dan tanpa *pocket* sehingga bobot akhir bergantung pada sampel terakhir.
- Hasil memakai satu split dan satu seed; ketidakpastian antar-split belum diukur.

**Pengembangan berikutnya:** ⟨mis. validasi silang, penambahan fitur interaksi, shuffle per epoch / pocket Perceptron, model non-linear (MLP)⟩.

## 10. Kesimpulan
⟨Jawab tujuan berdasarkan hasil, mis.:⟩
1. Kasus KFC berhasil dirumuskan sebagai klasifikasi dua kelas (High +1 / Low −1) tanpa kebocoran target.
2. Perceptron ⟨…⟩ (akurasi uji ⟨…⟩) dan ADALINE ⟨…⟩ (akurasi uji ⟨…⟩) pada konfigurasi η = 0,01 terstandardisasi.
3. ⟨Kesimpulan tentang separabilitas linear, pengaruh η/epoch/standardisasi, dan keterbatasan.⟩

## 11. Referensi dan Sumber Pendukung ⟨lengkapi⟩

**Artikel ilmiah & makalah dasar**
1. Rosenblatt, F. (1958). The perceptron: A probabilistic model for information storage and organization in the brain. *Psychological Review, 65*(6), 386–408. https://doi.org/10.1037/h0042519
2. Widrow, B., & Hoff, M. E. (1960). Adaptive switching circuits. *IRE WESCON Convention Record, Part 4*, 96–104. https://isl.stanford.edu/~widrow/papers/c1960adaptiveswitching.pdf
3. ⟨**Wajib:** minimal satu artikel ilmiah tentang metode atau kasus (mis. prediksi nilai/revenue transaksi restoran atau klasifikasi linear). Tulis sitasi + DOI/URL.⟩

**Kode / dokumentasi teknis**
4. Raschka, S. (tanpa tahun). Adaline: Adaptive Linear Neuron Classifier. Dokumentasi mlxtend. https://rasbt.github.io/mlxtend/user_guide/classifier/Adaline/ (diakses 6 Oktober 2026).

**Kontribusi tiap sumber, bagian yang diadaptasi, dan perubahan** ⟨isi, contoh: *Raschka (2026)* dipakai untuk memeriksa rumus pembaruan ADALINE batch; kami mengubah label menjadi −1/+1, mencatat J sebelum pembaruan, dan menambahkan penghentian saat NaN/inf.⟩

**Kontribusi anggota** ⟨isi per anggota⟩

In [ ]:
# Catat versi untuk README / requirements.txt
import sklearn, matplotlib
print("Python      :", sys.version.split()[0])
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib  :", matplotlib.__version__)
print("SEED        :", SEED)